# Ontology Agent Evaluation — Scores Overview

Analysis of AI coding agent performance across 4 biomedical ontologies,
comparing Claude Code and OpenAI Codex runtimes with multiple model tiers.

### Experimental design

We evaluate AI coding agents on their ability to resolve real GitHub issues in biomedical ontology repositories.
Each test case is a historical issue paired with the merged PR that resolved it. The agent receives the issue
context, starts from the same commit state as the original developer, and attempts to produce a fix.
We measure output quality using **metadiff** (F1, precision, recall) comparing the agent's diff against the human's.

Variables under test:
- **Runtime** (harness): Codex CLI, OpenCode, Pi — all using the same model (gpt-5.5)
- **Model**: gpt-5.4, gpt-5.5, claude-sonnet-4.5, claude-haiku-4.5
- **Ontology**: GO, Cell Ontology, Uberon, Mondo
- **Task difficulty**: simple, medium, hard
- **Skills**: with vs without (ablation)

In [ ]:
import os
# Ensure we're at repo root regardless of where jupyter runs
while not os.path.exists('analysis/scores.tsv'):
    os.chdir('..')
    if os.getcwd() == '/':
        raise RuntimeError('Could not find analysis/scores.tsv')
print(f'Working directory: {os.getcwd()}')

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ai4c_scribe.analysis import (
    load_scores, load_reviews, summary_by_model, summary_by_runtime,
    pivot_scores, rubric_summary, outcome_distribution, failure_mode_counts,
)

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
pd.set_option('display.precision', 3)

df = load_scores(Path('analysis/scores.tsv'))
reviews = load_reviews()
print(f'{len(df)} scored runs, {len(reviews)} qualitative reviews')
print(f'Ontologies: {sorted(df["ontology"].unique())}')
print(f'Models: {sorted(df["model"].unique())}')

---

We begin with aggregate views of performance, then drill down into controlled comparisons.
Note that raw model averages are **not directly comparable** because models were not run on
identical case sets. The paired analysis in Section 9 provides the rigorous comparison.

## 1. Model Comparison

In [ ]:
summary_by_model(df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
order = df.groupby('model')['f1'].mean().sort_values(ascending=False).index
sns.boxplot(data=df, x='model', y='f1', order=order, ax=ax)
sns.stripplot(data=df, x='model', y='f1', order=order, color='black', alpha=0.5, size=6, ax=ax)
ax.set_title('F1 Score Distribution by Model')
ax.set_xlabel('')
ax.set_ylabel('Metadiff F1')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('analysis/notebooks/fig_f1_by_model.png', dpi=150)
plt.show()

The runtime (harness) determines how the agent discovers skills, accesses tools, and manages permissions.
All runtimes follow the [Agent Skills](https://agentskills.io) open standard but differ in implementation details.
Codex uses its native `.agents/skills/` discovery; OpenCode reads both `.agents/` and `.claude/` directories;
Claude Code uses `.claude/skills/` with a dedicated Skill tool for invocation.

## 2. Runtime Comparison (Claude Code vs Codex)

In [ ]:
summary_by_runtime(df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Box plot by runtime
sns.boxplot(data=df, x='runtime', y='f1', ax=axes[0])
sns.stripplot(data=df, x='runtime', y='f1', color='black', alpha=0.5, ax=axes[0])
axes[0].set_title('F1 by Runtime')
axes[0].set_ylabel('Metadiff F1')

# By runtime and difficulty
sns.barplot(data=df, x='difficulty', y='f1', hue='runtime',
            order=['simple', 'medium', 'hard'], ax=axes[1])
axes[1].set_title('F1 by Difficulty and Runtime')
axes[1].set_ylabel('Mean F1')
axes[1].legend(title='Runtime')

plt.tight_layout()
plt.savefig('analysis/notebooks/fig_f1_by_runtime.png', dpi=150)
plt.show()

Performance varies dramatically across ontologies. GO has the highest scores because its agent config
includes 8 well-crafted skills (term-obsoletion, reaction, design-pattern, etc.) developed over months of
iteration. Cell Ontology and Uberon use OWL format with complex axiom patterns that agents find harder to edit correctly.

## 3. Ontology × Model Heatmap

In [ ]:
pivot = pivot_scores(df, rows='ontology', cols='model')
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Mean F1'})
ax.set_title('Mean F1 Score: Ontology × Model')
ax.set_ylabel('')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('analysis/notebooks/fig_heatmap_ont_model.png', dpi=150)
plt.show()

Task types reflect different ontology editing operations. Obsoletion is the most procedural (add metadata,
remove axioms, set replaced_by). New term requests (NTR) require understanding hierarchy placement and
writing definitions. Axiom repair requires identifying and fixing logical errors — the hardest for agents.

## 4. Task Type Analysis

In [ ]:
pivot_scores(df, rows='case_type', cols='runtime')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
task_order = df.groupby('case_type')['f1'].mean().sort_values(ascending=False).index
sns.barplot(data=df, x='case_type', y='f1', hue='runtime', order=task_order, ax=ax)
ax.set_title('Mean F1 by Task Type and Runtime')
ax.set_xlabel('Task Type')
ax.set_ylabel('Mean F1')
ax.legend(title='Runtime')
plt.tight_layout()
plt.savefig('analysis/notebooks/fig_f1_by_task_type.png', dpi=150)
plt.show()

The skills ablation tests whether the structured procedural knowledge in `.agents/skills/` SKILL.md files
actually improves performance. We created `-noskills` config variants that strip all skills, leaving only
the base AGENTS.md instructions.

**Key finding**: Claude Code completely fails without skills (F1=0.000 on all cases), while Codex shows
minimal degradation. This reflects architectural differences: Claude Code's permission system requires
skills for `allowed-tools` grants, while Codex runs with full access regardless.

## 5. Skills Ablation Study

In [ ]:
ablation = df[df['agent_config_tag'].isin(['v8', 'v8-noskills', 'v9', 'v2', 'v2-noskills', 'v3'])].copy()
ablation['has_skills'] = ~ablation['agent_config_tag'].str.contains('noskills')

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=ablation, x='runtime', y='f1', hue='has_skills', ax=ax)
ax.set_title('Skills Ablation: F1 With vs Without Skills')
ax.set_ylabel('Mean F1')
ax.legend(title='Skills Available', labels=['No Skills', 'With Skills'])
plt.tight_layout()
plt.savefig('analysis/notebooks/fig_skills_ablation.png', dpi=150)
plt.show()

print('Detail:')
pivot_scores(ablation, rows='agent_config_tag', cols='runtime')

## 6. All Scored Runs

In [ ]:
cols = ['ontology', 'issue_number', 'case_type', 'difficulty',
        'agent_config_tag', 'model', 'runtime', 'f1', 'precision', 'recall']
df[cols].sort_values(['ontology', 'issue_number', 'model']).style.background_gradient(
    subset=['f1', 'precision', 'recall'], cmap='RdYlGn', vmin=0, vmax=1
)

## 7. Qualitative Reviews

In [ ]:
if len(reviews) > 0:
    print(f'{len(reviews)} reviews loaded')
    display(rubric_summary(reviews))
else:
    print('No reviews yet')

In [ ]:
if len(reviews) > 0:
    rubric_cols = ['instruction_following', 'correctness', 'completeness',
                   'scope_discipline', 'methodology', 'overall']
    cols_present = [c for c in rubric_cols if c in reviews.columns]
    if cols_present:
        fig, ax = plt.subplots(figsize=(10, 5))
        melted = reviews.melt(id_vars=['model'], value_vars=cols_present,
                             var_name='rubric', value_name='score')
        sns.barplot(data=melted, x='rubric', y='score', hue='model', ax=ax)
        ax.set_title('Rubric Scores by Model')
        ax.set_ylabel('Score (1-5)')
        ax.set_xlabel('')
        ax.set_ylim(0, 5.5)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
        plt.tight_layout()
        plt.savefig('analysis/notebooks/fig_rubric_scores.png', dpi=150)
        plt.show()

In [ ]:
if len(reviews) > 0:
    print('Outcome distribution:')
    display(outcome_distribution(reviews))
    print('\nFailure modes:')
    display(failure_mode_counts(reviews))

## 8. Precision vs Recall

In [ ]:
nonzero = df[df['f1'] > 0].copy()

fig, ax = plt.subplots(figsize=(8, 8))
sns.scatterplot(data=nonzero, x='recall', y='precision', hue='runtime',
                style='difficulty', s=120, alpha=0.8, ax=ax)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.2)
ax.set_title('Precision vs Recall (non-zero runs)')
ax.set_xlabel('Recall (fraction of human changes reproduced)')
ax.set_ylabel('Precision (fraction of agent changes that match human)')

# Add F1 contours
import numpy as np
for f1_val in [0.2, 0.4, 0.6, 0.8]:
    r = np.linspace(0.01, 1, 100)
    p = (f1_val * r) / (2 * r - f1_val)
    mask = (p > 0) & (p <= 1)
    ax.plot(r[mask], p[mask], 'gray', alpha=0.15)
    ax.annotate(f'F1={f1_val}', xy=(1.0, (f1_val * 1.0) / (2 * 1.0 - f1_val)),
               fontsize=8, color='gray', alpha=0.5)

plt.tight_layout()
plt.savefig('analysis/notebooks/fig_precision_recall.png', dpi=150)
plt.show()

---

The preceding sections show descriptive patterns. This section provides the rigorous statistical test:
**does the harness affect performance when model, skills, and test cases are held constant?**

We use a paired design: for each test case run on both Codex and OpenCode with gpt-5.5, we compute
the F1 difference. A paired t-test and bootstrap confidence interval then test whether this difference
is significantly different from zero.

## 9. Statistical Analysis: Harness Comparison

With n=250 scored runs (69 paired cases between Codex and OpenCode on gpt-5.5),
we can now test whether the agent harness affects performance when the model,
skills, and test cases are held constant.

In [ ]:
from scipy import stats

# Paired comparison: Codex vs OpenCode on shared cases
df['case'] = df['ontology'] + '#' + df['issue_number'].astype(str)
codex_by_case = df[df['runtime'] == 'codex'].groupby('case')['f1'].mean()
opencode_by_case = df[df['runtime'] == 'opencode'].groupby('case')['f1'].mean()
shared_cases = codex_by_case.index.intersection(opencode_by_case.index)

c = codex_by_case.loc[shared_cases].values
o = opencode_by_case.loc[shared_cases].values
n_pairs = len(c)
diff = o - c

t_stat, p_val = stats.ttest_rel(o, c)

# Bootstrap CI
rng = np.random.default_rng(42)
boot = np.array([diff[rng.integers(0, n_pairs, n_pairs)].mean() for _ in range(10000)])
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
cohens_d = diff.mean() / diff.std(ddof=1)

print(f'Paired comparison: Codex vs OpenCode (gpt-5.5, same cases)')
print(f'  n = {n_pairs} paired cases')
print(f'  Codex mean F1:    {c.mean():.3f} (std={c.std():.3f})')
print(f'  OpenCode mean F1: {o.mean():.3f} (std={o.std():.3f})')
print(f'  Mean difference:  {diff.mean():+.3f}')
print(f'  Paired t({n_pairs-1}) = {t_stat:.3f}, p = {p_val:.4f}')
print(f'  95% bootstrap CI: [{ci_low:.3f}, {ci_high:.3f}]')
print(f'  Cohen\'s d = {cohens_d:.3f}')
print(f'  Significant at α=0.05: {"YES" if p_val < 0.05 else "NO"}')

In [ ]:
# Visualize the paired difference
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Paired scatter
ax = axes[0]
ax.scatter(c, o, alpha=0.6, s=40)
lims = [0, 1.05]
ax.plot(lims, lims, 'k--', alpha=0.3, label='y=x')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Codex F1')
ax.set_ylabel('OpenCode F1')
ax.set_title(f'Paired F1 (n={n_pairs} cases)\np={p_val:.3f}')
ax.legend()

# Panel B: Distribution of differences
ax = axes[1]
ax.hist(diff, bins=20, alpha=0.7, edgecolor='black')
ax.axvline(x=0, color='red', linestyle='--', label='No difference')
ax.axvline(x=diff.mean(), color='blue', linestyle='-', label=f'Mean={diff.mean():.3f}')
ax.axvspan(ci_low, ci_high, alpha=0.15, color='blue', label=f'95% CI')
ax.set_xlabel('F1 difference (OpenCode - Codex)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Paired Differences')
ax.legend(fontsize=9)

# Panel C: By difficulty
ax = axes[2]
diff_by_difficulty = []
labels = []
for d in ['simple', 'medium', 'hard']:
    sub = df[df['difficulty'] == d]
    c_sub = sub[sub['runtime'] == 'codex'].groupby('case')['f1'].mean()
    o_sub = sub[sub['runtime'] == 'opencode'].groupby('case')['f1'].mean()
    shared = c_sub.index.intersection(o_sub.index)
    if len(shared) >= 3:
        diffs = o_sub.loc[shared].values - c_sub.loc[shared].values
        diff_by_difficulty.append(diffs)
        labels.append(f'{d}\n(n={len(shared)})')
ax.boxplot(diff_by_difficulty, labels=labels)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.set_ylabel('F1 difference (OpenCode - Codex)')
ax.set_title('Effect by Difficulty')

plt.tight_layout()
plt.savefig('analysis/notebooks/fig_harness_comparison.png', dpi=150)
plt.show()

**Interpretation**: The left panel shows most points above the diagonal (OpenCode > Codex).
The center panel shows the difference distribution shifted right of zero, with the 95% CI
excluding zero. The right panel reveals the effect is concentrated in medium-difficulty tasks —
simple tasks hit a performance ceiling where both harnesses succeed, and hard tasks hit a floor
where neither performs well enough for the harness to make a difference.

The effect size (d=0.285) is small by conventional standards but meaningful in practice:
on a typical medium-difficulty case, switching from Codex to OpenCode improves F1 by ~7 percentage points.

---

The following tables provide the complete results suitable for inclusion in a manuscript.
Note that Claude Code has fewer runs (n=15) because OAuth rate limits constrained its evaluation
during this study period. A future analysis should include balanced Claude Code runs.

## 10. Summary Table for Paper

Complete results across all experimental conditions.

In [ ]:
# Summary table
summary = df.groupby(['runtime', 'ontology']).agg(
    n=('f1', 'count'),
    mean_f1=('f1', 'mean'),
    std_f1=('f1', 'std'),
    nonzero_rate=('f1', lambda x: (x > 0).mean()),
).round(3)
print('=== Results by Runtime × Ontology ===')
display(summary)
print()
print(f'Total runs: {len(df)}')
print(f'Success rate (F1 > 0): {(df["f1"] > 0).mean()*100:.0f}%')
print(f'Mean F1 (all): {df["f1"].mean():.3f}')
print(f'Mean F1 (non-zero only): {df[df["f1"]>0]["f1"].mean():.3f}')

In [ ]:
# Heatmap: Runtime × Ontology
pivot = df.pivot_table(values='f1', index='ontology', columns='runtime', aggfunc='mean')
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Mean F1'})
ax.set_title(f'Mean F1: Ontology × Runtime (n={len(df)} runs)')
ax.set_ylabel('')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('analysis/notebooks/fig_heatmap_runtime_ontology.png', dpi=150)
plt.show()